# Clase 4 — Práctica Guiada
## ¿Quiénes son los que se van?
### Maestría en Fintech · ITBA · 2026

---

**Esta notebook la resolvemos juntos en clase.** Después vas a tener una segunda notebook
—la de Práctica Individual— con preguntas nuevas sobre las mismas tablas, para resolver solo.

**Las preguntas de hoy:**
> *¿Qué segmento pone la plata? Y los clientes que dejaron de operar en 2024, ¿quiénes son?*

En la Clase 2 trabajamos con **clientes**. En la Clase 3, con **transacciones**. Cada tabla contestaba su
parte, y las dos clases terminaron con una pregunta que ninguna podía contestar sola. Hoy las cruzamos.

---
## Antes de empezar: guardá tu copia

Abriste esta notebook desde el link del curso, así que **nadie pisa el trabajo de nadie**.
Pero lo que escribas acá **no se guarda**: si cerrás la pestaña, se pierde.

Para quedarte con tu trabajo: **`Archivo`** → **`Guardar una copia en Drive`**.
Te sugerimos el nombre **`Clase4_Guiada_TuNombre`**.

---
## Setup

Ejecutá la celda con **`Shift + Enter`**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

print('Librerías cargadas correctamente')

---
# Paso 1 · Tres tablas, tres preguntas

Hoy no cargamos un CSV: cargamos **tres**. Cada uno responde una pregunta distinta sobre el mismo negocio.

| Tabla | Una fila es… | Responde |
|---|---|---|
| `clientes` | un cliente | ¿Quién es? |
| `tarjetas` | una tarjeta | ¿Qué tiene? |
| `transacciones` | un movimiento | ¿Qué hace? |

In [ ]:
# Las tablas se descargan solas desde el repo del curso.
DATOS = 'https://raw.githubusercontent.com/camilojaure/itba-pad/main/datasets/'

clientes      = pd.read_csv(DATOS + 'clientes.csv')
tarjetas      = pd.read_csv(DATOS + 'tarjetas.csv')
transacciones = pd.read_csv(DATOS + 'transacciones.csv')

print(f'clientes:      {len(clientes):>6,} filas')
print(f'tarjetas:      {len(tarjetas):>6,} filas')
print(f'transacciones: {len(transacciones):>6,} filas')

In [ ]:
clientes.head(3)

In [ ]:
tarjetas.head(3)

In [ ]:
transacciones.head(3)

**Buscá la columna que se repite en las tres:** `cliente_id`. Esa es la **clave**, la que permite cruzarlas.

Antes de unir nada, una pregunta que parece tonta y no lo es: **¿cuántas veces aparece cada cliente en cada tabla?**

In [ ]:
print(f"clientes:      {clientes['cliente_id'].nunique():>5,} clientes distintos en {len(clientes):>6,} filas")
print(f"tarjetas:      {tarjetas['cliente_id'].nunique():>5,} clientes distintos en {len(tarjetas):>6,} filas")
print(f"transacciones: {transacciones['cliente_id'].nunique():>5,} clientes distintos en {len(transacciones):>6,} filas")

**Cómo se lee:**

- En `clientes`, cada cliente aparece **una sola vez**: `cliente_id` es la **clave primaria** de esa tabla
- En `tarjetas` y `transacciones`, el mismo cliente aparece **muchas veces**: es una relación **uno a muchos**
- Y fijate los totales: **950** clientes tienen tarjeta y **905** operaron. De los 1.000, **faltan algunos**. Guardá ese dato para más adelante

> Saber cuántas filas por clave tiene cada tabla es lo que te permite predecir **cuántas filas deberían salir** de un join.
> Si no lo sabés antes, no vas a notar cuando salga mal.

---
# Paso 2 · El primer join: ¿qué segmento pone la plata?

La pregunta es simple. El problema es que **el gasto está en una tabla y el segmento en otra**:

In [ ]:
print('transacciones:', transacciones.columns.tolist())
print('clientes:     ', clientes.columns.tolist())

`pd.merge` hace lo mismo que un **BUSCARV** en Excel o un **JOIN** en SQL: a cada transacción le trae los datos
de su cliente, buscándolo por `cliente_id`.

```python
pd.merge(transacciones, clientes, on='cliente_id', how='left')
#        ↑ izquierda     ↑ derecha      ↑ la clave      ↑ qué hacer con lo que no matchea
```

**Antes de correrlo, predecí:** cada transacción tiene un solo cliente. ¿Cuántas filas deberían salir?

In [ ]:
df = pd.merge(transacciones, clientes, on='cliente_id', how='left', validate='many_to_one')

print(f'{len(transacciones):,} transacciones → {len(df):,} filas después del merge')
df.head(3)

**Mismas filas, más columnas.** Cada transacción ahora "sabe" el segmento, la provincia y el balance de su cliente.

> `validate='many_to_one'` es opcional, pero vale la pena: le decís a pandas *"del lado derecho cada cliente
> aparece una sola vez"*. Si no fuera cierto, el merge frena con un error en vez de multiplicar filas en silencio.
> Es la regla de contar filas, pero automática.

Ahora sí, la pregunta:

In [ ]:
seg = df.groupby('segmento').agg(
    clientes_que_operan=('cliente_id', 'nunique'),
    transacciones=('transaccion_id', 'count'),
    monto_M=('monto_ars', 'sum'),
)
seg['monto_M'] = (seg['monto_M'] / 1e6).round(1)
seg['% del gasto'] = (seg['monto_M'] / seg['monto_M'].sum() * 100).round(1)

# La participación en la cartera sale de la tabla de clientes, no de la unida
seg['% de la cartera'] = (clientes['segmento'].value_counts(normalize=True) * 100).round(1)

seg.sort_values('monto_M', ascending=False)

**Lectura de negocio:**

- **Premium** es menos de 1 de cada 5 clientes y pone **más de la mitad del gasto**
- **Joven** es 1 de cada 4 clientes y pone **menos del 5%**
- Retail tiene la mayor cantidad de transacciones, pero chicas

> ¿Se acuerdan del cierre de la Clase 2? El segmento que más se va (Joven) es el que menos plata pone en riesgo.
> Ahora lo vemos en el gasto: **no todos los clientes pesan lo mismo**, y esa respuesta no estaba en ninguna de las dos tablas por separado.

---
# Paso 3 · El error silencioso

La tabla unida es cómoda: tiene todo. Usémosla para una pregunta más: **¿cuánto balance tiene la cartera?**

In [ ]:
print(f"Balance total según la tabla de clientes: ${clientes['balance_ars'].sum() / 1e6:>8,.0f} M")
print(f"Balance total según la tabla unida:       ${df['balance_ars'].sum() / 1e6:>8,.0f} M")

**Pregunta para el grupo antes de seguir:** ¿cuál está bien? ¿Y de dónde salen los otros 25.000 millones?

In [ ]:
# Miremos un solo cliente en la tabla unida
un_cliente = df[df['cliente_id'] == 'C0001']
print(f"C0001 aparece {len(un_cliente)} veces: una por cada transacción")
un_cliente[['cliente_id', 'nombre', 'balance_ars', 'fecha', 'monto_ars']].head()

> **El join repitió el balance del cliente en cada una de sus transacciones.** Al sumar, cada balance se contó
> tantas veces como compras hizo ese cliente. C0001 se contó 41 veces; toda la cartera, **37 veces su balance real**.
>
> Lo peligroso es que **no hay error**. El código corre, el número sale con formato prolijo y está mal.
> Es el error más frecuente con joins, y el más caro, porque el número llega a una presentación.

**La regla:** después de cada merge, **contá las filas** y preguntate si tiene sentido.
Y cada dato se mide en su tabla: el balance es del **cliente**, la plata gastada es de la **transacción**.
Si necesitás los dos en la misma tabla, **primero resumís y después unís** (lo hacemos en el Paso 5).

---
# Paso 4 · Inner vs. left: lo que no matchea

Ahora cruzamos **clientes** con **tarjetas**. La diferencia entre los tipos de join está en qué pasa con
las filas que **no encuentran pareja** del otro lado.

| `how=` | Qué devuelve | En Excel era |
|---|---|---|
| `'inner'` | Solo los que están en **las dos** tablas | Borrar las filas donde BUSCARV dio `#N/A` |
| `'left'` | **Todos** los de la izquierda, con lo que haya de la derecha | Dejar los `#N/A` a la vista |

> **Si no aclarás `how`, pandas usa `inner`.** Estás descartando filas sin enterarte.

In [ ]:
con_inner = pd.merge(clientes, tarjetas, on='cliente_id', how='inner')
con_left  = pd.merge(clientes, tarjetas, on='cliente_id', how='left')

print(f"clientes:  {len(clientes):,} filas")
print(f"inner  →   {len(con_inner):,} filas · {con_inner['cliente_id'].nunique():,} clientes")
print(f"left   →   {len(con_left):,} filas · {con_left['cliente_id'].nunique():,} clientes")

**Dos cosas para leer:**

1. **Hay más filas que clientes** en los dos casos: un cliente con tres tarjetas aparece tres veces (uno a muchos, otra vez)
2. **El inner perdió 50 clientes.** El left los conserva, con las columnas de la tarjeta vacías (`NaN`)

Después de un left join, un `NaN` significa *"no encontré pareja del otro lado"*. ¿Quiénes son?

In [ ]:
sin_tarjeta = con_left[con_left['tarjeta_id'].isna()]

print(sin_tarjeta['segmento'].value_counts())
print()
print(f"Churn de los clientes sin tarjeta: {sin_tarjeta['churn'].mean():.0%}")
print(f"Churn de toda la cartera:          {clientes['churn'].mean():.0%}")

> **Esos 50 no son un problema de datos: son el hallazgo.** Clientes sin ningún producto de tarjeta, mayoría
> Joven, que se van **dos veces y media más** que el promedio. Es una lista para una campaña comercial.
>
> **El inner los habría hecho desaparecer sin avisar.** Por eso, ante la duda, empezá con `left` y mirá los huecos.

---
# Paso 5 · La vista del cliente: una fila por cliente

Lo que la industria llama **vista 360**: una tabla donde cada cliente aparece una vez, con todo lo que sabemos de él.
La receta es la regla del Paso 3: **primero resumís cada tabla por cliente, después unís.**

In [ ]:
# La fecha viene como texto: la convertimos (como en la Clase 3) y sacamos el mes
transacciones['fecha'] = pd.to_datetime(transacciones['fecha'])
transacciones['mes'] = transacciones['fecha'].dt.month

# 1) Resumir: una fila por cliente en cada tabla
gasto_cliente = transacciones.groupby('cliente_id').agg(
    monto_2024=('monto_ars', 'sum'),
    cant_transacciones=('transaccion_id', 'count'),
    ultimo_mes=('mes', 'max'),
).reset_index()

tarjetas_cliente = tarjetas.groupby('cliente_id').agg(
    cant_tarjetas=('tarjeta_id', 'count'),
).reset_index()

# 2) Unir: todo contra clientes, con left, para no perder a nadie
vista = (clientes
         .merge(tarjetas_cliente, on='cliente_id', how='left')
         .merge(gasto_cliente, on='cliente_id', how='left'))

print(f'{len(vista):,} filas — una por cliente')
vista.head()

Los clientes sin tarjeta o sin transacciones quedaron con `NaN`. Acá ese vacío significa **cero**, así que lo completamos:

In [ ]:
columnas = ['cant_tarjetas', 'monto_2024', 'cant_transacciones']
vista[columnas] = vista[columnas].fillna(0)

print(f"Balance total, ahora sí:            ${vista['balance_ars'].sum() / 1e6:,.0f} M")
print(f"Clientes sin ninguna transacción:   {(vista['cant_transacciones'] == 0).sum()}")

**El balance volvió a dar bien**, porque cada cliente aparece una sola vez.

Y aparecen **95 clientes que no operaron en todo 2024**: los 50 sin tarjeta y **45 que tienen tarjeta y nunca la usaron**.
Ninguna de las tablas por separado te daba esa lista.

> `ultimo_mes` queda en `NaN` para esos 95, y está bien: no tienen último mes. No lo completamos con cero.

---
# Paso 6 · ⏱ La pregunta que quedó abierta en la Clase 3

La Clase 3 terminó con esto: el gasto se duplicó en el año, pero los clientes que operan **bajaron de 781 en enero a 727 en diciembre**.
Contábamos IDs en la tabla de transacciones, sin saber quiénes eran.

Ahora podemos preguntarle a cada transacción si su cliente **se quedó o se fue**:

In [ ]:
operan = (transacciones
          .merge(clientes[['cliente_id', 'churn']], on='cliente_id', how='left')
          .groupby(['mes', 'churn'])['cliente_id'].nunique()
          .unstack(fill_value=0))

operan.columns = ['Se quedan', 'Se fueron']
operan['Total'] = operan['Se quedan'] + operan['Se fueron']
operan

In [ ]:
operan[['Se quedan', 'Se fueron']].plot(marker='o', linewidth=2, color=['steelblue', 'firebrick'])
plt.title('Clientes que operan cada mes, según si se quedaron o se fueron — 2024')
plt.ylabel('Clientes distintos')
plt.xlabel('Mes')
plt.xticks(range(1, 13))
plt.show()

**La lectura que da sentido a las tres clases:**

- Los clientes que **se quedan** no bajaron: **subieron**, de 675 a 727
- Los que **se fueron** pasaron de 106 a **cero**. En diciembre no operó ninguno
- **Toda la caída que vimos en la Clase 3 son clientes que se estaban yendo**

> La base que se queda está sana y crece. El problema es de retención, no de actividad.
> Y se veía venir mes a mes: los que se iban **operaban cada vez menos antes de irse**.

Esta respuesta no estaba en `clientes` ni en `transacciones`. Estaba **en el cruce**.

---
# Paso 7 · ¿Con cuánta anticipación se ve?

Si los que se van dejan de operar antes de irse, la pregunta de negocio es **cuánto antes**.
Usamos la vista del Paso 5: para cada cliente que se fue, ¿cuál fue su último mes con una transacción?

In [ ]:
se_fueron = vista[vista['churn'] == 1]

se_fueron['ultimo_mes'].value_counts().sort_index().plot(kind='bar', color='firebrick')
plt.title('Último mes con transacciones de los clientes que se fueron')
plt.xlabel('Mes')
plt.ylabel('Clientes')
plt.xticks(rotation=0)
plt.show()

print(f"Mediana del último mes: {se_fueron['ultimo_mes'].median():.0f} (julio)")
print(f"Nunca operaron en 2024: {se_fueron['ultimo_mes'].isna().sum()}")

> **La mitad de los que se fueron ya no operaba después de julio.** Entre que un cliente deja de usar la tarjeta
> y aparece como baja pasan meses. Ese es el tiempo que tiene el equipo de retención para actuar,
> y hoy no lo está usando porque mira la baja, no la actividad.

---
# Paso 8 · El caso: ¿a quién llama mañana el equipo de retención?

Si dejar de operar es la señal, buscamos a los que **siguen siendo clientes** pero **ya dejaron de operar**:
los que están hoy donde estaban en julio los que se fueron.

In [ ]:
dormidos = vista[(vista['churn'] == 0) & (vista['ultimo_mes'] < 12)]

print(f"{len(dormidos)} clientes siguen con nosotros pero no operaron en diciembre")
print(f"Balance en juego: ${dormidos['balance_ars'].sum() / 1e6:,.1f} M")
print()
print(dormidos['segmento'].value_counts())

In [ ]:
# La lista, ordenada por lo que está en juego
(dormidos
 .sort_values('balance_ars', ascending=False)
 [['cliente_id', 'nombre', 'segmento', 'balance_ars', 'ultimo_mes', 'monto_2024']]
 .head(10))

**Arriba de todo: cuatro Premium, con $4,6M de balance entre los cuatro.** Son la primera tanda de llamados.

### Del análisis a la recomendación — tres bullets, no una tabla

1. **El diagnóstico:** los clientes que operan bajaron de 781 a 727, pero los que se quedan **crecieron** de 675 a 727. La caída es 100% clientes que se fueron
2. **La señal:** los que se fueron dejaron de operar meses antes (mediana: julio). La inactividad avisa antes que la baja
3. **La acción:** hoy hay **29 clientes** que no operaron en diciembre y siguen con nosotros ($8,0M de balance). Llamar primero a los **4 Premium** y armar una alerta mensual con esta misma consulta

**Escribamos los tres bullets juntos, en voz alta, con los números que acabamos de sacar.**

---
## Hasta acá la práctica guiada

**Lo que hiciste hoy:** cruzaste tres tablas con `merge`, predijiste cuántas filas tenían que salir, viste la diferencia
entre `inner` y `left`, encontraste un error que no tira error, armaste una vista con una fila por cliente
y contestaste la pregunta que quedó abierta en la Clase 3.

**Lo que sigue:** abrí la notebook **`Clase_4_Practica_Individual.ipynb`**.
Mismas tablas, preguntas nuevas, y esta vez lo resolvés vos con Gemini de copiloto.

> Si querés quedarte con tu trabajo: **`Archivo → Guardar una copia en Drive`**.